In [1]:
import pandas as pd

In [10]:
#Ques 1: Z-score normalise sales within each category and add a percentile rank column per category. 

df_p = pd.DataFrame({ "product": ["A","B","C","D","E","F","G","H"], "category": ["Electronics","Electronics","Clothing","Clothing", "Electronics","Clothing","Electronics","Clothing"], "sales": [1200,800,500,700,1500,600,900,450] })
df_p['normalized'] = df_p.groupby('category')['sales'].transform(lambda x: (x-x.mean())/x.std())
df_p['pct_rank'] = df_p.groupby('category')['sales'].transform(lambda x: x.rank(pct=True)*100)
display(df_p)

,product,category,sales,normalized,pct_rank
0,A,Electronics,1200,0.316228,75.0
1,B,Electronics,800,-0.948683,25.0
2,C,Clothing,500,-0.563735,50.0
3,D,Clothing,700,1.240216,100.0
4,E,Electronics,1500,1.264911,100.0
5,F,Clothing,600,0.338241,75.0
6,G,Electronics,900,-0.632456,50.0
7,H,Clothing,450,-1.014722,25.0


In [22]:
#Ques 2: Aggregate sales at region and store levels, then find stores with |z-score| > 2 vs region mean.

df_h = pd.DataFrame({ "region": ["North","North","North","South","South","South","North","South"], "store": ["S1","S1","S2","S3","S3","S4","S2","S4"], "sales": [1000,1500,3000,1200,1100,900,2800,5000] }) 
display(df_h)

df_sales_store = df_h.groupby(['region', 'store'], as_index = False)['sales'].sum().rename(columns={'sales': 'total_store_sales'})

df_sales_region = df_sales_store.groupby('region')['total_store_sales'].agg(region_mean='mean', region_std = 'std').reset_index()

df_merged = df_sales_store.merge(df_sales_region, on='region')

df_merged['z-score'] = (df_merged['total_store_sales'] - df_merged['region_mean'])/df_merged['region_std']

df_outliers = df_merged[df_merged['z-score'].abs() > 2]

display(df_merged.round(2))

display(df_outliers)

,region,store,sales
0,North,S1,1000
1,North,S1,1500
2,North,S2,3000
3,South,S3,1200
4,South,S3,1100
5,South,S4,900
6,North,S2,2800
7,South,S4,5000


,region,store,total_store_sales,region_mean,region_std,z-score
0,North,S1,2500,4150.0,2333.45,-0.71
1,North,S2,5800,4150.0,2333.45,0.71
2,South,S3,2300,4100.0,2545.58,-0.71
3,South,S4,5900,4100.0,2545.58,0.71


,region,store,total_store_sales,region_mean,region_std,z-score


In [ ]:
# Ques 3: For each product, compute cumulative sales, diff from previous period, % change, and 3-period moving average.

base = pd.DataFrame({ "date": pd.date_range("2024-01-01", periods=12, freq="ME"), "product": ["A"]*12, "sales": [100,120,110,130,125,140,135,150,145,160,155,170] }) 
b2 = base.copy() 
b2["product"] = "B" 
b2["sales"] = [80,90,85,95,100,105,110,115,120,125,130,135] 
df_w = pd.concat([base,b2],ignore_index=True).sort_values(["product","date"])

g = df_w.groupby('product')['sales']

df_w['cumulative_sum'] = g.cumsum()
df_w['previous_sales'] = g.shift(1)
df_w['diff_sales'] = df_w['sales'] - df_w['previous_sales']
df_w['p_change'] = g.pct_change() * 100
df_w['moving_average'] = df_w['sales'].rolling(3).mean()
df_w["moving_avg3"] = g.transform(lambda x: x.rolling(3).mean())
display(df_w)

,date,product,sales,cumulative_sum,previous_sales,diff_sales,p_change,moving_average,moving_avg3
0,2024-01-31,A,100,100,NaN,NaN,NaN,NaN,NaN
1,2024-02-29,A,120,220,100.0,20.0,20.000000,NaN,NaN
2,2024-03-31,A,110,330,120.0,-10.0,-8.333333,110.000000,110.000000
3,2024-04-30,A,130,460,110.0,20.0,18.181818,120.000000,120.000000
4,2024-05-31,A,125,585,130.0,-5.0,-3.846154,121.666667,121.666667
5,2024-06-30,A,140,725,125.0,15.0,12.000000,131.666667,131.666667
6,2024-07-31,A,135,860,140.0,-5.0,-3.571429,133.333333,133.333333
7,2024-08-31,A,150,1010,135.0,15.0,11.111111,141.666667,141.666667
8,2024-09-30,A,145,1155,150.0,-5.0,-3.333333,143.333333,143.333333
9,2024-10-31,A,160,1315,145.0,15.0,10.344828,151.666667,151.666667


In [63]:
#Ques 4: Find customers who: (1) purchased in last 30 days, (2) total spending > $1000, (3) avg order value > $100.

end_date = pd.Timestamp("2023-12-20") 
tx = pd.DataFrame({ "customer_id": [1,1,2,2,2,3,3,4,4,5], "order_date": pd.date_range(end_date-pd.Timedelta(days=60), periods=10, freq="7D"), "amount": [150,200,800,120,900,300,400,500,600,500] }) 
tx["days_ago"] = (end_date - tx['order_date']).dt.days
display(tx)
tx = tx.groupby('customer_id').agg(
    total_spending = ('amount', 'sum'),
    avg_order_value = ('amount', 'mean'),
    total_orders = ('amount', 'count'),
    days_since_last = ('days_ago', 'min')    
).reset_index()
display(tx)

df_output = tx[(tx['days_since_last'] <= 30) & (tx['total_spending'] > 1000) & (tx['avg_order_value'] > 100)]
display(df_output)


,customer_id,order_date,amount,days_ago
0,1,2023-10-21,150,60
1,1,2023-10-28,200,53
2,2,2023-11-04,800,46
3,2,2023-11-11,120,39
4,2,2023-11-18,900,32
5,3,2023-11-25,300,25
6,3,2023-12-02,400,18
7,4,2023-12-09,500,11
8,4,2023-12-16,600,4
9,5,2023-12-23,500,-3


,customer_id,total_spending,avg_order_value,total_orders,days_since_last
0,1,350,175.000000,2,53
1,2,1820,606.666667,3,32
2,3,700,350.000000,2,18
3,4,1100,550.000000,2,4
4,5,500,500.000000,1,-3


,customer_id,total_spending,avg_order_value,total_orders,days_since_last
3,4,1100,550.0,2,4
